#### Extract raw text using Pymupdf

In [1]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import TextLoader
import json
import unicodedata
import os 
from dotenv import load_dotenv

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

#### Create the template to extract the JSON

In [2]:
template = """
You are an expert resume parser. Your job is to extract structured information from the raw resume text provided below.

RESUME TEXT:
{extracted_text}

Extract the following fields and return ONLY valid JSON matching this schema:
{
  "name": string or null,
  "contact": {"email": string or null, "phone": string or null, "location": string or null, "linkedin": string or null},
  "summary": string or null,
  "skills": [string] or null,
  "experience": [{"company": string or null, "title": string or null, "start_date": string or null, "end_date": string or null, "responsibilities": [string]}] or null,
  "education": [{"degree": string or null, "institution": string or null, "year": string or null}] or null,
  "projects": [{"name": string or null, "description": string or null, "tech": [string]}] or null,
  "certifications": [string] or null
}

---

FEW SHOT EXAMPLES:

Example 1 — Software Engineer Resume:

Input:
John Doe
john.doe@gmail.com | +1-9876543210 | New York, USA | linkedin.com/in/johndoe

Profile:
Software Engineer with 3 years of experience building scalable backend systems.

Skills: Python, Django, PostgreSQL, Docker, AWS, Git

Experience:
Software Engineer, Google
06/2021 – Present | New York, USA
- Designed and deployed RESTful APIs serving 1M+ requests/day
- Reduced system latency by 40% through query optimization

Education:
B.Tech - Computer Science, MIT | 2017 – 2021

Projects:
Smart Inventory System
Built an automated inventory tracking tool using IoT sensors and Python.
Tech: Python, MQTT, PostgreSQL, Docker

Certifications:
AWS Certified Developer – Associate

Output:
{{
  "name": "John Doe",
  "contact": {{
    "email": "john.doe@gmail.com",
    "phone": "+1-9876543210",
    "location": "New York, USA",
    "linkedin": "linkedin.com/in/johndoe"
  }},
  "summary": "Software Engineer with 3 years of experience building scalable backend systems.",
  "skills": ["Python", "Django", "PostgreSQL", "Docker", "AWS", "Git"],
  "experience": [
    {{
      "company": "Google",
      "title": "Software Engineer",
      "start_date": "06/2021",
      "end_date": "Present",
      "responsibilities": [
        "Designed and deployed RESTful APIs serving 1M+ requests/day",
        "Reduced system latency by 40% through query optimization"
      ]
    }}
  ],
  "education": [
    {{
      "degree": "B.Tech - Computer Science",
      "institution": "MIT",
      "year": "2017 – 2021"
    }}
  ],
  "projects": [
    {{
      "name": "Smart Inventory System",
      "description": "Built an automated inventory tracking tool using IoT sensors and Python.",
      "tech": ["Python", "MQTT", "PostgreSQL", "Docker"]
    }}
  ],
  "certifications": ["AWS Certified Developer – Associate"]
}}
---

RULES (strictly follow these):
1. If a field is missing from the resume, set it to null. Do NOT guess or assume.
2. Do NOT mix projects with experiences. Experience = paid roles at companies. Projects = personal/academic/side work.
3. Only include a project if it has both a name AND a description. If description is missing, exclude the project entirely.
4. Responsibilities must be extracted as-is from the resume. Do not rephrase or summarize them.
5. Return ONLY the JSON. No explanation, no preamble, no markdown code block, no extra text.
6. Dates must be in DD/MM/YYYY format wherever possible. If only year is available, use YYYY.
"""

In [3]:
llm_qwen = ChatGroq(
    model = 'qwen/qwen3-32b',
    temperature = 0 ,
    max_tokens= 4000)

In [4]:
def clean_text(text):
    # Normalize unicode characters to their closest ASCII equivalent
    text = unicodedata.normalize("NFKD", text)
    
    # Replace common unicode punctuation with plain equivalents
    replacements = {
        "\u2013": "-",   # en dash  →  -
        "\u2014": "-",   # em dash  →  -
        "\u2018": "'",   # left single quote
        "\u2019": "'",   # right single quote
        "\u201c": '"',   # left double quote
        "\u201d": '"',   # right double quote
        "\u2022": "",    # bullet point
        "\u00a0": " ",   # non-breaking space
        "\u00b7": ".",
        "\u2192": "->",
        "\u223c": "~",
        "|": "",         # pipe separator
    }
    
    for unicode_char, replacement in replacements.items():
        text = text.replace(unicode_char, replacement)
    
    return text

#### Function to extract JSON

In [5]:

def get_json(file_path , llm, template = template):
    loader = PyMuPDFLoader(file_path)
    
    # load the file
    docs = loader.load()
    # Combine all pages into one string
    res = "\n".join([doc.page_content for doc in docs])
    # clean the text before passing it into LLM
    res = clean_text(res)

    formatted_prompt = template.replace("{extracted_text}",res) + "\n/no_think"

    response = llm.invoke(formatted_prompt).content

    json_str = response.split("</think>")[-1].strip()
    parsed = json.loads(json_str)

    return parsed
    

In [ ]:
# get_json(r"..\data\MeetLad_Resume.pdf" , llm_qwen)

{
    "name": "Meet Lad",
    "contact": {
        "email": "ladmeet27@gmail.com",
        "phone": "+918097408972",
        "location": "Mumbai, Maharashtra, India",
        "linkedin": null
    },
    "summary": "Data professional with 2.5 years experience in data analysis, management, and predictive modeling, and 4.5+ years of total experience in software development and data-driven solutions. Skilled in Python, SQL, Tableau, Apache Superset, and AWS, with a proven track record of building and deploying data-driven solutions. Seeking a role in a data-driven organization to apply analytical and technical expertise for impactful business outcomes.",
    "skills": [
        "Python",
        "SQL",
        "AWS (ECS, EC2, DYNAMO DB, S3, CLOUDFRONT, IAM)",
        "Numpy",
        "Pandas",
        "Scikit-learn",
        "SciPy",
        "Seaborn",
        "Boto3"
    ],
    "experience": [
        {
            "company": "The Yarn Bazaar",
            "title": "Data Analyst",
       

{'name': 'Meet Lad',
 'contact': {'email': 'ladmeet27@gmail.com',
  'phone': '+918097408972',
  'location': 'Mumbai, Maharashtra, India',
  'linkedin': None},
 'summary': 'Data professional with 2.5 years experience in data analysis, management, and predictive modeling, and 4.5+ years of total experience in software development and data-driven solutions. Skilled in Python, SQL, Tableau, Apache Superset, and AWS, with a proven track record of building and deploying data-driven solutions. Seeking a role in a data-driven organization to apply analytical and technical expertise for impactful business outcomes.',
 'skills': ['Python',
  'SQL',
  'AWS (ECS, EC2, DYNAMO DB, S3, CLOUDFRONT, IAM)',
  'Numpy',
  'Pandas',
  'Scikit-learn',
  'SciPy',
  'Seaborn',
  'Boto3'],
 'experience': [{'company': 'The Yarn Bazaar',
   'title': 'Data Analyst',
   'start_date': '04/2022',
   'end_date': 'Present',
   'responsibilities': ['Machine Learning & Time Series Analysis: Developed and deployed a mach

#### Below function is created for single resume upload via UI

In [6]:
def process_single_resume(file_path, llm, output_folder="output_jsons_resume", force=False):
    """
    Process one resume PDF and add it to all_resumes.json.

    If the resume filename already exists in all_resumes.json, it is skipped
    unless force=True.
    """

    os.makedirs(output_folder, exist_ok=True)

    pdf_file = os.path.basename(file_path)
    combined_output = os.path.join(output_folder, "all_resumes.json")

    if os.path.exists(combined_output):
        with open(combined_output, "r", encoding="utf-8") as f:
            all_results = json.load(f)
    else:
        all_results = {}

    if pdf_file in all_results and not force:
        return {
            "status": "skipped",
            "message": f"{pdf_file} is already processed.",
            "data": all_results[pdf_file]
        }

    parsed = get_json(file_path, llm)

    if not parsed:
        return {
            "status": "failed",
            "message": f"Could not parse {pdf_file}.",
            "data": None
        }

    output_file = os.path.join(
        output_folder,
        pdf_file.replace(".pdf", ".json")
    )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed, f, indent=4)

    all_results[pdf_file] = parsed

    with open(combined_output, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=4)

    return {
        "status": "processed",
        "message": f"{pdf_file} processed and added to all_resumes.json.",
        "data": parsed
    }


#### Process all the resumes at same time

In [7]:
def process_multiple_resumes(folder_path, llm, template=template, output_folder="output_jsons_resume"):
    """
    Process only new resumes from a folder.
    Already processed resumes are skipped.
    """

    pdf_files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".pdf")
    ]

    if not pdf_files:
        print("No PDF files found in the folder.")
        return {}

    already_processed = 0
    new_processed = 0
    failed = 0
    results = {}

    print(f"Found {len(pdf_files)} resumes.\n")

    for pdf_file in pdf_files:
        file_path = os.path.join(folder_path, pdf_file)

        result = process_single_resume(
            file_path=file_path,
            llm=llm,
            output_folder=output_folder,
            force=False
        )

        results[pdf_file] = result

        if result["status"] == "skipped":
            already_processed += 1
            print(f"Skipping already processed resume: {pdf_file}")

        elif result["status"] == "processed":
            new_processed += 1
            print(f"Processed new resume: {pdf_file}")

        else:
            failed += 1
            print(f"Failed: {pdf_file}")

    print("\nProcessing Summary")
    print(f"Already processed: {already_processed}")
    print(f"New resumes processed: {new_processed}")
    print(f"Failed: {failed}")

    return results


In [ ]:
# process_multiple_resumes("C:\Resume_Screening_Project\data" , llm_qwen, template = template, output_folder="C:\Resume_Screening_Project\output_jsons_resume")

Found 6 resumes.

Skipping already processed resume: Abhishek_Shaurya_Resume.pdf
Skipping already processed resume: Jay Kumar Behera_CV_DS.pdf
Skipping already processed resume: Komal_Kamble_1 (1).pdf
Skipping already processed resume: MeetLad_Resume.pdf
Skipping already processed resume: Prity-Kumari-Resume-DevOps-2025.pdf
Skipping already processed resume: SOUMYADEEP_SEN_CV.pdf

Processing Summary
Already processed: 6
New resumes processed: 0
Failed: 0


<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\komkambl\AppData\Local\Temp\ipykernel_30296\4104800597.py:1: SyntaxWarning: invalid escape sequence '\R'
  process_multiple_resumes("C:\Resume_Screening_Project\data" , llm_qwen, template = template, output_folder="C:\Resume_Screening_Project\output_jsons_resume")
C:\Users\komkambl\AppData\Local\Temp\ipykernel_30296\4104800597.py:1: SyntaxWarning: invalid escape sequence '\R'
  process_multiple_resumes("C:\Resume_Screening_Project\data" , llm_qwen, template = template, output_folder="C:\Resume_Screening_Project\output_jsons_resume")


{'Abhishek_Shaurya_Resume.pdf': {'status': 'skipped',
  'message': 'Abhishek_Shaurya_Resume.pdf is already processed.',
  'data': {'name': 'Abhishek Shaurya',
   'contact': {'email': 'abhishekshauryaoff@gmail.com',
    'phone': '+91-8109149513',
    'location': 'Bengaluru',
    'linkedin': 'linkedin.com/in/ashrya'},
   'summary': 'Product Owner with 3 years driving platform and financial system modernization across 90+ enterprise retail clients (NA, EMEA, APAC). Owns roadmap, backlog, and delivery across ETL re-architecture, reporting platform migration, and a 0->1 AI audit assistant - saving ~7,500 engineering hours annually and surfacing $4-5M in financial risk exposure. Targeting senior PM roles in B2B SaaS, fintech, or data/AI products.',
   'skills': ['Roadmap Ownership',
    'Feature Prioritization (RICE, MoSCoW)',
    'Platform Architecture',
    'PRD/BRD',
    'MVP Scoping',
    'Financial Systems Modeling',
    'User Research',
    'SQL',
    'Python',
    'Snowflake',
    'Mi

In [16]:

with open(r'..\output_jsons_resume\all_resumes.json','r', encoding='utf-8') as f:
    data = json.load(f)

print(len(data))

6


#### Extract the JD


In [58]:
jd_template = """
You are an expert at parsing Job Descriptions.
Please follow the rules mention STRICTLY.
Extract the following fields from the JD text below and return ONLY valid JSON.

JD TEXT:
{extracted_text}

Extract the following fields and return ONLY valid JSON matching this schema:
{
  "job_title": string or null,
  "company": string or null,
  "location": string or null,
  "employment_type": string or null,
  "experience_required": string or null,
  "notice_period": string or null,
  "job_summary": string or null,
  "skills_required": {
      "must_have": [string] or null,
      "good_to_have": [string] or null
  },
  "responsibilities": [{
        "category": string or null,
        "tasks": [string]}] or null,
  "professional_qualifications": {
      "must_have": [string] or null,
      "preferred": [string] or null
  },
  "education_qualification": string or null,
  "salary": string or null
}

---

FEW SHOT EXAMPLES:

Example 1 — Complete JD:

Input:
Senior DevOps Engineer — TechCorp, Bangalore
Employment Type: Full-Time | Experience: 4-6 years | Notice Period: 30 days or less

Job Overview:
We are looking for a Senior DevOps Engineer to manage and scale our cloud infrastructure,
automate deployments and ensure high availability of our systems.

Must Have Skills: Docker, Kubernetes, Jenkins, AWS, Linux
Good to Have: Terraform, Ansible, Prometheus

Required Tasks:
- Design and manage CI/CD pipelines
- Monitor system health and uptime
- Collaborate with development teams on deployments
- Perform root cause analysis for production incidents

Required Qualifications:
- 4+ years of hands-on experience in DevOps
- AWS Solutions Architect certification

Preferred Qualifications:
- Experience with multi-cloud environments
- Exposure to Site Reliability Engineering practices

Education: B.Tech in Computer Science or related field

Salary: 18-25 LPA

Output:
{
  "job_title": "Senior DevOps Engineer",
  "company": "TechCorp",
  "location": "Bangalore",
  "employment_type": "Full-Time",
  "experience_required": "4-6 years",
  "notice_period": "30 days or less",
  "job_summary": "We are looking for a Senior DevOps Engineer to manage and scale our cloud infrastructure, automate deployments and ensure high availability of our systems.",
  "skills_required": {
    "must_have": ["Docker", "Kubernetes", "Jenkins", "AWS", "Linux"],
    "good_to_have": ["Terraform", "Ansible", "Prometheus"]
  },
  "responsibilities": [
    "Design and manage CI/CD pipelines",
    "Monitor system health and uptime",
    "Collaborate with development teams on deployments",
    "Perform root cause analysis for production incidents"
  ],
  "professional_qualifications": {
    "must_have": [
      "4+ years of hands-on experience in DevOps",
      "AWS Solutions Architect certification"
    ],
    "preferred": [
      "Experience with multi-cloud environments",
      "Exposure to Site Reliability Engineering practices"
    ]
  },
  "education_qualification": "B.Tech in Computer Science or related field",
  "salary": "18-25 LPA"
}

RULES:
1. Return ONLY valid JSON. No explanation, no preamble, no markdown blocks.
2. If a field is missing, set it to null. Do not make anything up.
3. SYNONYM RULE FOR JOB SUMMARY:
   The job_summary field can appear under many different headings in a JD.
   All of the following mean the same thing — map them all to "job_summary":
   "Summary", "Overview", "Job Overview", "Job Description", 
   "About the Role", "Role Overview", "Position Summary", "About the Job"
4. SYNONYM RULE FOR RESPONSIBILITIES:
   The responsibilities field can appear under many different headings.
   All of the following mean the same thing — map them all to "responsibilities":
   "Responsibilities", "Key Responsibilities", "Required Tasks", 
   "What You Will Do", "Your Role", "Day to Day", "Key Tasks",
   "Duties", "What We Expect", "Role & Responsibilities"
5. SYNONYM RULE FOR PROFESSIONAL QUALIFICATIONS:
   Split professional qualifications into must_have and preferred:
   - must_have  → "Required Qualifications", "Minimum Requirements", 
                  "Must Have", "Basic Qualifications", "You Must Have"
   - preferred  → "Preferred Qualifications", "Nice to Have", 
                  "Good to Have", "Bonus Points", "Preferred Skills",
                  "It Would Be Great If"
6. SYNONYM RULE FOR SKILLS:
   - must_have     → "Required Skills", "Must Have Skills", "Technical Requirements",
                     "Core Skills", "Key Skills", "Essential Skills"
   - good_to_have  → "Good to Have", "Nice to Have", "Bonus Skills",
                     "Preferred Skills", "Optional Skills"
   If NO distinction is made between skill types, put ALL skills under 
   must_have and set good_to_have to null.
7. EDUCATION QUALIFICATION:
   Extract only ONE string for education (the minimum degree requirement).
   Look for headings like: "Education", "Educational Qualification", 
   "Degree", "Academic Background", "Minimum Education".
   Do not list multiple education entries — just the primary requirement.
8. Responsibilities and qualifications must be extracted as-is.
   Do not rephrase, summarize, or merge bullet points.
9. Dates, experience, and salary must be extracted exactly as written in the JD.
10. If experience_required is greater than 1 years and employment_type is extracted as null, go ahead and replace it with "Full-Time".
11. If responsibilities dont have a category, only return tasks.
12. If professional_qualifications is exactly similar as education_qualification, put professional_qualifications as null.
"""

In [84]:
def extract_jd_json(file_path, llm, template=jd_template):
    loader = TextLoader(file_path, encoding='utf-8')
    doc = loader.load()

    formatted_prompt = jd_template.replace("{extracted_text}",doc[0].page_content)

    response = llm_qwen.invoke(formatted_prompt).content

    json_str = response.split("</think>")[-1].strip()
    parsed = json.loads(json_str)
    return parsed

In [ ]:
def extract_multiple_jd(folder_path, llm=llm_qwen, output_folder="output_json_jd"):
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Extract the txt_files in a list
    txt_files = [ t for t in os.listdir(folder_path) if t.endswith(".txt")]

    all_results = {}

    if not txt_files:
        print("No txt files found in the folder.")
        return {}

    print(f"Found {len(txt_files)} txt. Processing...\n")

    for txt_file in txt_files:
        file_path = os.path.join(folder_path, txt_file)
        print(f"Processing: {file_path}")

        parsed = extract_jd_json(file_path,llm= llm_qwen)

        if parsed:
            output_file = os.path.join(output_folder, txt_file.replace(".txt",".json") )
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(parsed, f, indent=4)
            print(f"✅ Saved: {output_file}")

            all_results[txt_file] = parsed
        else:
            print(f"Skipped {txt_file} due to parsing error.\n")

    if all_results:
        combined_output = os.path.join(output_folder, "all_jd.json")
        with open(combined_output, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=4)
        print(f"\n✅ All done! Combined JSON saved at: {combined_output}")
    else:
        print("\n⚠️ No JDs were successfully parsed. Combined JSON not saved.")
            
    return all_results